In [8]:
#Ref
import pandas as pd
import glob
import numpy as np
import plotly.express as px
import re

path = '/Users/jiakai/Desktop/SURF/code/_spirit/'
# n_cycles = 120
dim = 4

# Pattern to match all relevant CSV files
file_pattern = path + f"negative_field_cycle_v2_dim{dim}_anisotropy1.5_ncycles*_gamma1e-07_H_high6.0_H_low*.csv"
# negative_field_cycle_v2_dim4_anisotropy1.5_ncycles1200_gamma1e-07_H_high6.0_H_low4.0

# List all matching files
csv_files = glob.glob(file_pattern)

for f in csv_files:
    print(f)

# Extract H_low values
H_lows = []
for path in csv_files:
    match = re.search(r'H_low([0-9.]+)\.csv', path)
    if match:
        h_low = float(match.group(1))
        H_lows.append(h_low)

print(H_lows)

/Users/jiakai/Desktop/SURF/code/_spirit/negative_field_cycle_v2_dim4_anisotropy1.5_ncycles1200_gamma1e-07_H_high6.0_H_low6.0.csv
/Users/jiakai/Desktop/SURF/code/_spirit/negative_field_cycle_v2_dim4_anisotropy1.5_ncycles1200_gamma1e-07_H_high6.0_H_low4.0.csv
/Users/jiakai/Desktop/SURF/code/_spirit/negative_field_cycle_v2_dim4_anisotropy1.5_ncycles1200_gamma1e-07_H_high6.0_H_low2.0.csv
/Users/jiakai/Desktop/SURF/code/_spirit/negative_field_cycle_v2_dim4_anisotropy1.5_ncycles1200_gamma1e-07_H_high6.0_H_low5.0.csv
[6.0, 4.0, 2.0, 5.0]


In [10]:
import plotly.graph_objects as go

# n_cycles = 1200  # Adjust if needed
fig = go.Figure()

# Sort by numeric H_low extracted from the filename
csv_files = sorted(csv_files, key=lambda f: float(re.search(r'H_low([0-9.]+)\.csv', f).group(1)))

for file in csv_files:
    # Extract H_low from filename
    match = re.search(r'H_low([0-9.]+)\.csv', file)
    if not match:
        continue
    H_low_val = float(match.group(1))

    # Load CSV
    df_all = pd.read_csv(file)

    # Group and compute mean & SEM
    grouped = df_all.groupby(['i', 'Ht']).agg(
        chi_mean=('chi', 'mean'),
        chi_std=("chi", lambda x: x.std(ddof=1) / np.sqrt(len(x)))
    ).reset_index()

    # grouped['chi_sem'] = grouped['chi_std'] / (n_cycles ** 0.5)

    # Add trace for this H_low
    fig.add_trace(go.Scatter(
        x=grouped["i"],
        y=grouped["chi_mean"],
        error_y=dict(
            type="data",
            array=grouped["chi_std"],
            visible=True
        ),
        mode="lines+markers",
        name=f"H_low = {H_low_val}" if H_low_val != 6.0 else "Ref (H_low = H_high)",
        hovertemplate=f"H_low={H_low_val}<br>i=%{{x}}<br>χ=%{{y:.3f}}<extra></extra>"
    ))

# Final layout
fig.update_layout(
    title="Chi vs MCS Step",
    xaxis_title="MCS Step (i)",
    yaxis_title="Mean χ",
    template="plotly_white",
    legend=dict(
        x=1,
        y=1,
        xanchor="right",
        yanchor="top"
    )
)

fig.write_html(f'negative_field_cycle_dim{dim}_H_high6.html')
fig.show()